### **Decorators:** Decorators Python ka ek aisa powerful feature hain jo aapko kisi function ya class ke behavior ko modify ya extend karne ki permission deta hai, woh bhi bina unke original code ko touch kiye.

#### **1. Function-based Decorators:** Ek function-based decorator basically ek function hota hai jo argument ke roop mein ek doosra function leta hai, uske andar ek inner function (wrapper) banata hai jo extra behavior add karta hai, aur phir us wrapper ko return kar deta hai.

In [17]:
def say_hello(name):
    print(f"Hello {name}!")

obj = say_hello
obj("Ram")

Hello Ram!


In [16]:
# Assigning a function to a variable
def greet(n):
    return f"Hello, {n}!"
say_hi = greet 
print(say_hi("Alex"))

# Passing a function as an argument
def apply(f, v):
    return f(v)
res = apply(say_hi, "Elon")
print(res) 

Hello, Alex!
Hello, Elon!


In [1]:
# Returning a function from another function
def make_mult(f):
    def mult(x):
        return x * f
    return mult
dbl = make_mult(2)
print(dbl(5))

10


In [6]:
def my_decorator(func):
    def wrapper(*args, **kwargs):
        print("--> Function call hone se pahleka logic (pre-processing)")
        result = func(*args, **kwargs)
        print("--> Function call hone ke baad ka logic (Post-processing)")
        return result
    return wrapper

@my_decorator
def say_hello(name):
    print(f"Hello {name}!")

say_hello('Ram')

--> Function call hone se pahleka logic (pre-processing)
Hello Ram!
--> Function call hone ke baad ka logic (Post-processing)


#### **Explanation in Hinglish:** Jab aap @my_decorator likhte hain, toh Python say_hello function ko my_decorator ke andar pass kar deta hai. wrapper function execute hota hai, jo pehle apna message print karta hai, phir original say_hello ko call karta hai, aur baad mein apna post-processing logic run karta hai.

#### **2. Method Decorators:** Special decorators used for methods inside a class. They work like function decorators but handle the self parameter for instance methods.

In [ ]:
def method_decorator(func):
    def wrapper(self, *args, **kwargs):
        print("Before method execution")
        res = func(self, *args, **kwargs)
        print("After method execution")
        return res
    return wrapper

class MyClass:
    @staticmethod
    @method_decorator
    def say_hello(self, name):
        print(f"Hello {name}!")
obj = MyClass()
obj.say_hello("Ram")

Before method execution Ram
Hello Ram!
After method execution


#### **3. Class-based Decorators:** Jab aapko decorator ke andar state (data/history) maintain karni ho (jaise kitni baar function call hua yeh track karna), tab hum function ki jagah class-based decorators ka use karte hain. Iske liye class mein __ call __ dunder method ka use hota hai.

In [1]:
class CountCalls:
    def __init__(self, func):
        self.func = func
        self.num_calls = 0

    def __call__(self, *args, **kwargs):
        self.num_calls += 1
        print(f"yeh function {self.num_calls} baar call ho chuka hai.")
        return self.func(*args, **kwargs)

@CountCalls
def say_hi():
    print("Hi")

say_hi()
say_hi()

yeh function 1 baar call ho chuka hai.
Hi
yeh function 2 baar call ho chuka hai.
Hi


#### Explanation in Hinglish: Jab class ko decorator ki tarah use karte hain, toh __ init __ function ko store karta hai, aur jab bhi decorated function call hota hai, toh class ka __ call __ method trigger ho jata hai jisme hum counter maintain kar pa rahe hain.

#### **4. Parameterized Decorators:** Kabhi-kabhi humein decorator ko khud kuch arguments pass karne hote hain (jaise @repeat(num_times=3)). Iske liye humein teen levels ke functions ki zarurat padti hai: ek outer function jo parameters leta hai, ek jo decorator banata hai, aur ek jo wrapper hota hai.

In [ ]:
def repeat(num_times):
    def decorator_repeat(func):
        def wrapper(*args, **kwargs):
            for _ in range(num_times):
                result = func(*args, **kwargs)
            return result
        return wrapper
    return decorator_repeat

@repeat(num_times = 3)
def greet(name):
    print(f"Namaste", {name})

greet("Ram")

Namaste {'Ram'}
Namaste {'Ram'}
Namaste {'Ram'}


#### Explanation in Hinglish: Yahan repeat(num_times=3) ek factory function ki tarah kaam karta hai jo arguments ko accept karke actual decorator (decorator_repeat) return karta hai.

### How works decorators in background?
Jab aap code ke upar @decorator_name likhte hain, toh Python ke background mein ek chhota sa magic hota hai jise Syntactic Sugar kehte hain.
Aaiye dekhte hain ki background mein exactly kya hota hai aur parameters kaise pass hote hain:
#### 1. Syntactic Sugar ka Asli Roop (Background Transformation)
Jab aap yeh likhte hain:
```
@my_decorator
def add(a, b):
    return a + b
```
Python background mein isko bilkul is tarah rewrite (convert) kar deta hai:
```
def add(a, b):
    return a + b

add = my_decorator(add)
```
#### Matlab kya hai?
Background mein Python khud-ba-khud original add function ko as an argument uthakar my_decorator ke andar pass kar deta hai. Aur jo naya function (wrapper) return hota hai, usko wapas add variable mein assign kar deta hai. Isiliye jab aap baad mein add(5, 3) call karte hain, toh aap asal mein decorator ke andar ka wrapper function call kar rahe hote hain!

#### 2. Normal Function ke Parameters wrapper tak kaise jaate hain? (*args aur **kwargs)
Jab decorator ke andar function banta hai, toh hum usme *args aur **kwargs lagate hain:

```
def my_decorator(func):
    def wrapper(*args, **kwargs):  # Yeh sabhi parameters ko catch karta hai
        print("Function call hone se pehle extra logic...")
        
        # Yahan parameters ko original function ko bhej diya jata hai
        result = func(*args, **kwargs) 
        
        print("Function call hone ke baad extra logic...")
        return result
    return wrapper
```
* *args aur **kwargs ka role: Jab aap add(5, 3) call karte hain, toh wrapper(5, 3) call hota hai. Yeh *args aur **kwargs un arguments (5 aur 3) ko apne andar safely capture kar lete hain aur phir func(*args, **kwargs) ke zariye original add function ko pass kar dete hain.
* Agar aap *args aur **kwargs nahi lagayenge, toh decorator ko pata nahi chalega ki function ko kitne parameters diye gaye hain, aur error aa jayega.